In [0]:
import datetime
from pyspark import Row

user_list = [
    {
        'id': 1,
        'f_name': 'John',
        'l_name': 'Maddocks',
        'email': 'abc@xyz.com',
        'phone_numbers': Row(mobile='+1 234 567 8901', home='+1 234 444 5556'),
        'courses': [3],
        'is_customer': True,
        'amount_paid': 1001.22,
        'customer_from': datetime.date(2020, 2, 1),
        'last_updated': datetime.datetime(2021, 3, 4, 10, 40, 25)
    },
    {
        'id': 2,
        'f_name': 'Bob',
        'l_name': 'Miller',
        'email': 'abc@xyz.com',
        'phone_numbers': Row(mobile='+1 234 567 9999', home='+1 234 444 7777'),
        'courses': [2, 4],
        'is_customer': False,
        'amount_paid': 1000.5,
        'customer_from': datetime.date(2021, 6, 1),
        'last_updated': datetime.datetime(2022, 3, 4, 10, 40, 25)
    },
        {
        'id': 3,
        'f_name': 'Chris',
        'l_name': 'Miller',
        'email': 'abc@xyz.com',
        'phone_numbers': Row(mobile='+1 234 444 7777', home=None),
        'courses': [],
        'is_customer': True,
        'amount_paid': 1004.5,
        'customer_from': datetime.date(2021, 6, 1),
        'last_updated': datetime.datetime(2022, 3, 4, 10, 40, 25)
    },
        {
        'id': 4,
        'f_name': 'christy',
        'l_name': 'Marsh',
        'email': 'abc@xyz.com',
        'phone_numbers': Row(mobile=None, home=None),
        'is_customer': True,
        'amount_paid': 1004.5,
        'customer_from': datetime.date(2021, 6, 1),
        'last_updated': datetime.datetime(2022, 3, 4, 10, 40, 25)
    }
]

df = spark.createDataFrame(user_list)
df.show()

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import col, concat, lit

df.select(col('id'),
          'f_name',
          'l_name',
         concat(col('f_name'), lit(', '), col('l_name')).alias('full_name')).show()

In [0]:
from pyspark.sql.functions import date_format

temp_df = df.select(col('id'), date_format('customer_from', 'yyyyMMdd').alias('cust_start_dt'))
temp_df.show()

In [0]:
temp_df.printSchema()

In [0]:
# https://spark.apache.org/docs/3.5.3/sql-ref-datetime-pattern.html

In [0]:
customer_data = df.select(col('id'),
                          date_format('customer_from', 'yyyy').cast('int').alias('Year'),
                          date_format('customer_from', 'MM').cast('int').alias('Month'),
                          date_format('customer_from', 'dd').cast('int').alias('Day')
                        )
customer_data.show()

In [0]:
df.select(col('amount_paid'),(col('amount_paid') + 25).alias('amount paid_increment')).show()

In [0]:
df.select('id', 'f_name', 'l_name').withColumn('full_name', concat('f_name', lit(', '), 'l_name')).show()

In [0]:
#renaming a single column

df.select('id', 'f_name', 'l_name').withColumnRenamed('id', 'user_id').show()

In [0]:
# renaming multiple columns

(
    df.select('id', 'f_name', 'l_name')
    .withColumnRenamed('id', 'user_id')
    .withColumnRenamed('f_name', 'first_name')
    .withColumnRenamed('l_name', 'last_name')
    .show()
)

In [0]:
df.select(
    col('id').alias('user_id'), 
    col('f_name').alias('first_name'), 
    col('l_name').alias('last_name'),
    concat(col('f_name'), lit(', '), col('l_name')).alias('full_name')
    ).show()

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import size

df.select('id', 'courses').withColumn('course_count', size('courses')).show()

In [0]:
employees = [(1, 'Scott', 'Tiger', 1000.0, 'united states', '+1 123 456 7890', '123 45 6789'),
             (2, 'Henry', 'Ford', 1250.0, 'India', '+91 234, 567 8901', '456 78 9123'),
             (3, 'Nick', 'Junior', 750.0, 'united KINGDOM', '+44 111 111 1111', '222 33 4444'),
             (4, 'Bill', 'Gomes', 1500.00, 'AUSTRALIA', '+61 987 654 3210', '789 12 6118')]

schema='''emp_id INT, f_name STRING, l_name STRING, salary FLOAT, nationality STRING, phone_number STRING, ssn STRING'''
emp_df = spark.createDataFrame(data=employees, schema=schema)

emp_df.show()

In [0]:
from pyspark.sql import functions as F

In [0]:
# this will work
#F.col('salary') creates a Column object referencing the 'salary' column.it is a column object
#F.lit(0.2) creates a Column object representing the constant value 0.2.

emp_df.withColumn('bonus', F.col('salary') * F.lit(0.2)).show()

In [0]:

from pyspark.sql import functions as F

emp_df.withColumns(
    {
        "bonus": F.col("salary") * F.lit(0.2),
        "bonus_c": F.col("salary") * F.lit(0.1)
    }
).show()


In [0]:
from pyspark.sql import functions as F

emp_df.withColumns(
    {
        "bonus": F.col("salary") * F.lit(0.2),
        "bonus_c": F.col("salary") * F.lit(0.1)
    }
).withColumnsRenamed({
    "bonus": "bonus_tmp",
    "bonus_c": "bonus_c_tmp"
}).show()


### String Manipulations

In [0]:
# using lower() function
# mentioned column name in withColumn is same as original column name
# it will overright orignal column in the dataframe

emp_df.withColumn('nationality', F.lower('nationality')).show()

In [0]:
# using upper() function
# creating new column

emp_df.withColumn('updated_nationality', F.upper('nationality')).show()

In [0]:
emp_df.withColumn('nationality', F.initcap('nationality')).show()

In [0]:
emp_df.withColumn('len', F.length('nationality')).show()

In [0]:
emp_df.withColumn('short_name', F.substring('f_name', 1, 4)).show()

In [0]:
emp_df.withColumn('ssn_components',F.split('ssn', ' ')).show()

In [0]:
# split results in Array type column

emp_df.withColumn('ssn_components',F.split('ssn', ' ')).printSchema()

In [0]:
# Try: trim, ltrim, rtrim, lpad, rpad, instr, etc.


### Datetime Manipulation

In [0]:
# creating dummy dataframe
#“dummy” DataFrame is a simple, single-column DataFrame often used for testing.
l = [('X', )]
df = spark.createDataFrame(l).toDF('dummy')
df.show()
# get current date
#print(df.select(F.current_date()).show(truncate=False)) # default/standard format #yyyy-MM-dd
df.select(F.current_date()).show(truncate=False)
# get current date and time
#print(df.select(F.current_timestamp()).show(truncate=False)) # default/standard format #yyyy-MM-dd HH:mm:ss.SSS
df.select(F.current_timestamp()).show(truncate=False) 
df.show()


In [0]:
df.select(
    F.current_date().alias('current_date'),
    F.year(F.current_date()).alias('year'),
    F.month(F.current_date()).alias('month'),
    F.weekofyear(F.current_date()).alias('weekofyear'),
    F.dayofyear(F.current_date()).alias('dayofyear'),
    F.dayofmonth(F.current_date()).alias('dayofmonth'),
    F.dayofweek(F.current_date()).alias('dayofweek')
).show()

In [0]:
df.select(
    F.current_timestamp().alias('current_timestamp'),
    F.year(F.current_timestamp()).alias('year'),
    F.month(F.current_timestamp()).alias('month'),
    F.weekofyear(F.current_timestamp()).alias('weekofyear'),
    F.dayofyear(F.current_timestamp()).alias('dayofyear'),
    F.dayofmonth(F.current_timestamp()).alias('dayofmonth'),
    F.dayofweek(F.current_timestamp()).alias('dayofweek'),
    F.hour(F.current_timestamp()).alias('hour'),
    F.minute(F.current_timestamp()).alias('minute'),
    F.second(F.current_timestamp()).alias('second'),
                                
).show(truncate=False)

In [0]:
# converting string into date

df.select(F.to_date(F.lit('20210302'), 'yyyyMMdd').alias('to_date')).show()

In [0]:
# to_date
#yyyyDDD is a date format where:
#yyyy = year (e.g., 2021)
#DDD = day of the year (e.g., 061 is the 61st day)

df.select(F.to_date(F.lit('2021061'), 'yyyyDDD').alias('to_date')).show() # julian day / three digit day of the year

In [0]:
# to_date

df.select(F.to_date(F.lit('02/03/2021'), 'dd/MM/yyyy').alias('to_date')).show()

In [0]:
df.select(F.to_timestamp(F.lit('02-Mar-2021'), 'dd-MMM-yyyy').alias('to_date')).show()

In [0]:
df.select(F.to_timestamp(F.lit('02-Mar-2021 17:30:15'), 'dd-MMM-yyyy HH:mm:ss').alias('to_date')).show()

In [0]:
# date-time formats

# yyyy: 4 digit year
# MM : 2 digit month
# MMM : 3 letter abbreviated month
# MMMM : full month name
# dd : day of the month
# DDD : day of the year/ julian day
# HH : hours (24 hours format)
# hh : hours (12 hours format)
# mm : minutes
# ss : seconds
# SSS : milliseconods
# EE : 3 letter abbreviated weekday

In [0]:
datetimes = [
    ("2021-01-28", '2021-01-28 10:00:00.123'),
    ("2022-02-18", '2022-02-18 08:08:08.993'),
    ("2023-02-25", '2023-02-25 11:59:59.343'),
    ("2023-02-23", '2023-02-23 00:00:00.000'),
]

dt_tm_df = spark.createDataFrame(datetimes, schema='date STRING, time STRING')

dt_tm_df.show(truncate=False)

In [0]:
dt_tm_df.printSchema()

In [0]:
dt_tm_df. \
    withColumn('date_add_date', F.date_add('date', 10)). \
    withColumn('date_add_time', F.date_add('time', 10)). \
    withColumn('date_sub_date', F.date_sub('date', 10)). \
    withColumn('date_sub_time', F.date_sub('time', 10)). \
show(truncate=False)

In [0]:
dt_tm_df. \
    withColumn('date_add_date', F.date_add('date', 10)). \
    withColumn('date_add_time', F.date_add('time', 10)). \
    withColumn('date_sub_date', F.date_sub('date', 10)). \
    withColumn('date_sub_time', F.date_sub('time', 10)).printSchema()

In [0]:
# gives diff between 2 given dates in days
#F.datediff(end, start) => end - start
#start => past date =>positive
#start => end => 0
#start => future date => negative
dt_tm_df. \
    withColumn('date_diff_date', F.datediff(F.current_date(), 'date')). \
    withColumn('date_diff_time', F.datediff(F.current_date(), 'time')). \
show()

In [0]:
# Try: months_between, add_months, trunc, date_trunc, etc.

### Manipulating with Null values in Spark Dataframes

In [0]:
employees = [(1, 'Scott', 'Tiger', 1000.0, 10, 'united states', '+1 123 456 7890', '123 45 6789'),
             (2, 'Henry', 'Ford', 1250.0, None, 'India', '+91 234, 567 8901', '456 78 9123'),
             (3, 'Nick', 'Junior', 750.0, '', 'united KINGDOM', '+44 111 111 1111', '222 33 4444'),
             (4, 'Bill', 'Gomes', 1500.00, 10, 'AUSTRALIA', '+61 987 654 3210', '789 12 6118')]

schema = '''emp_id INT, f_name STRING, l_name STRING, salary FLOAT, bonus STRING, nationality STRING, phone_number STRING, ssn STRING'''

emp_df = spark.createDataFrame(employees, schema=schema)

emp_df.show()

In [0]:
from pyspark.sql import functions as F
emp_df_clean = emp_df.withColumn(
    'bonus_c',
    F.when(F.col('bonus') == '', None).otherwise(F.col('bonus'))
)

emp_df_clean.show()

emp_df_clean.withColumn(
    'bonus1',
    F.coalesce(F.col('bonus_c').cast('int'), F.lit(0))
).show()

In [0]:
# calculate payment

from pyspark.sql import functions as F

emp_df.withColumn(
    "payment",
    F.col("salary") +
    (     F.col("salary") *
        F.coalesce(F.expr("try_cast(bonus as int)"),
            F.lit(0)
        ) / 100
    )
).show()



In [0]:

from pyspark.sql import functions as F


emp_df.withColumn(
    "payment",
    F.col("salary") * (
        1 + F.when(F.col("bonus") != "", F.col("bonus").cast("int"))
              .otherwise(0) / 100
    )
).show()


In [0]:
# complete solution with expr

emp_df.withColumn('bonus', F.expr("nvl(nullif(bonus, ''),0)")).show()

In [0]:
# retrun true if empty of null and false otherwise

from pyspark.sql import functions as F
emp_df.withColumn('getting_bonus', F.expr("nvl2(nullif(bonus, ''),true, false)")).show()



======================================== END ===========================================